# 08 Final Training and Evaluation

Train the final PPE detector using the locked architecture and final training configuration, then evaluate it once on the untouched test set.


## Purpose

Notebook 08 is the first pipeline step allowed to use the test split. Candidate architecture selection happened in Notebook 06, and augmentation/config selection happened in Notebook 07. Test metrics here are for honest final reporting only, not for choosing another model or changing augmentation after the fact.


## Final Configuration Note

Notebook 07 may show that online augmentation alone performs best on the current easy test-like distribution. For real factory deployment, the final retrain intentionally prefers the full augmentation pipeline (`exp_D_full_pipeline`) because production CCTV can be harder: glare, blur, compression, night/IR, and camera variation matter more than the easiest validation result.


## 1. Setup

Find the `v2_pipeline` root, add it to `sys.path`, and set Windows/Jupyter safety variables before importing Ultralytics-facing helpers.


In [1]:
from pathlib import Path
import os
import shutil
import sys

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")


def find_v2_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate v2_pipeline root.")


try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

V2_ROOT = find_v2_root(NOTEBOOK_DIR)
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

print(f"v2_pipeline root: {V2_ROOT}")

v2_pipeline root: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline


Import project helpers and common analysis packages. Heavy YOLO logic stays in `src/training/` so this notebook remains readable.


In [2]:
import pandas as pd
import yaml

from src.training.train_yolo import train_final_model
from src.training.evaluate_yolo import evaluate_final_model, save_final_predictions
from src.training.export_model import export_final_model

## 2. Load Configuration

Read training, augmentation, and class-name settings. The final export format defaults to ONNX unless more formats are added in `training_config.yaml`.


In [3]:
CONFIG_DIR = V2_ROOT / "configs"
REPORTS_DIR = V2_ROOT / "reports" / "training"
FIGURES_DIR = V2_ROOT / "reports" / "figures"
FINAL_RUNS_DIR = V2_ROOT / "runs" / "final_model"
FINAL_WEIGHTS_DIR = V2_ROOT / "weights" / "final"

with (CONFIG_DIR / "training_config.yaml").open("r", encoding="utf-8") as file_handle:
    training_config = yaml.safe_load(file_handle)

with (CONFIG_DIR / "augmentation_config.yaml").open(
    "r", encoding="utf-8"
) as file_handle:
    augmentation_config = yaml.safe_load(file_handle)

with (CONFIG_DIR / "class_names.yaml").open("r", encoding="utf-8") as file_handle:
    class_config = yaml.safe_load(file_handle)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
FINAL_RUNS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

class_config

{'nc': 3, 'names': {0: 'person', 1: 'helmet', 2: 'vest'}}

## 3. Resolve Selected Architecture

Architecture is resolved from prior reports first, then config fallback. This uses validation-selected architecture information only; test metrics are not consulted.


In [4]:
def top_successful_model(report_path: Path) -> str | None:
    if not report_path.exists():
        return None
    report = pd.read_csv(report_path)
    if report.empty or "model_name" not in report.columns:
        return None
    if "status" in report.columns:
        report = report[report["status"].isin(["trained", "evaluated"])]
    if report.empty:
        return None
    return str(report.iloc[0]["model_name"])


ablation_model = top_successful_model(REPORTS_DIR / "ablation_ranking.csv")
candidate_model = top_successful_model(REPORTS_DIR / "candidate_model_ranking.csv")
config_model = training_config.get("selected_model")

if ablation_model:
    selected_model = ablation_model
    selected_model_source = "reports/training/ablation_ranking.csv"
elif candidate_model:
    selected_model = candidate_model
    selected_model_source = "reports/training/candidate_model_ranking.csv"
elif config_model:
    selected_model = str(config_model)
    selected_model_source = "configs/training_config.yaml:selected_model"
else:
    candidates = training_config.get("candidate_models", [])
    if not candidates:
        raise ValueError(
            "No selected model, candidate ranking, or candidate_models config is available."
        )
    selected_model = str(candidates[0])
    selected_model_source = "first candidate_models fallback"

print(f"Selected architecture: {selected_model}")
print(f"Architecture source: {selected_model_source}")

Selected architecture: yolov8n.pt
Architecture source: reports/training/ablation_ranking.csv


## 4. Resolve Final Experiment

The ablation winner is recorded for traceability. The final retrain uses `selected_experiment` from config, which is set to `exp_D_full_pipeline` to favor robustness under harder real-world CCTV conditions.


In [5]:
EXPERIMENT_YAMLS = {
    "exp_A_original_only": V2_ROOT / "data_exp_A_original_only.yaml",
    "exp_B_online_aug": V2_ROOT / "data_exp_B_online_aug.yaml",
    "exp_C_offline_aug": V2_ROOT / "data_exp_C_offline_aug.yaml",
    "exp_D_full_pipeline": V2_ROOT / "data_exp_D_full_pipeline.yaml",
}
ONLINE_AUG_EXPERIMENTS = {"exp_B_online_aug", "exp_D_full_pipeline"}

ablation_ranking_path = REPORTS_DIR / "ablation_ranking.csv"
ablation_top_experiment = None
if ablation_ranking_path.exists():
    ablation_ranking = pd.read_csv(ablation_ranking_path)
    if not ablation_ranking.empty and "experiment" in ablation_ranking.columns:
        successful = ablation_ranking
        if "status" in successful.columns:
            successful = successful[successful["status"].eq("trained")]
        if not successful.empty:
            ablation_top_experiment = str(successful.iloc[0]["experiment"])

preferred_experiment = training_config.get("selected_experiment", "exp_D_full_pipeline")
selected_experiment = str(
    preferred_experiment or ablation_top_experiment or "exp_D_full_pipeline"
)
if selected_experiment not in EXPERIMENT_YAMLS:
    raise ValueError(f"Unknown selected experiment: {selected_experiment}")

selected_data_yaml = EXPERIMENT_YAMLS[selected_experiment]
if not selected_data_yaml.exists():
    raise FileNotFoundError(f"Selected experiment YAML not found: {selected_data_yaml}")

selection_note = (
    f"Ablation top experiment: {ablation_top_experiment or 'not available'}. "
    f"Final retrain uses {selected_experiment} by config/user preference for real-world robustness."
)

print(selection_note)
print(f"Selected dataset YAML: {selected_data_yaml.relative_to(V2_ROOT)}")

Ablation top experiment: exp_B_online_aug. Final retrain uses exp_D_full_pipeline by config/user preference for real-world robustness.
Selected dataset YAML: data_exp_D_full_pipeline.yaml


## 5. Validate Dataset Paths

Before training, verify that the selected YOLO YAML points to train, validation, and test image folders. This notebook does not modify datasets.


In [6]:
def dataset_dirs_from_yaml(yaml_path: Path) -> dict[str, Path]:
    with yaml_path.open("r", encoding="utf-8") as file_handle:
        data_config = yaml.safe_load(file_handle)
    dataset_root = (yaml_path.parent / data_config["path"]).resolve()
    return {
        "root": dataset_root,
        "train_images": dataset_root / data_config["train"],
        "val_images": dataset_root / data_config["val"],
        "test_images": dataset_root / data_config["test"],
    }


dataset_dirs = dataset_dirs_from_yaml(selected_data_yaml)
dataset_check = pd.DataFrame(
    [
        {"path_type": key, "path": str(path), "exists": path.exists()}
        for key, path in dataset_dirs.items()
    ]
)
display(dataset_check)

if not dataset_check["exists"].all():
    raise FileNotFoundError(
        "Selected experiment dataset folders are missing. Re-run Notebook 05 first."
    )

,path_type,path,exists
0,root,C:\Github\smart-factory-safety-monitoring-syst...,True
1,train_images,C:\Github\smart-factory-safety-monitoring-syst...,True
2,val_images,C:\Github\smart-factory-safety-monitoring-syst...,True
3,test_images,C:\Github\smart-factory-safety-monitoring-syst...,True


## 6. Build Final Training Arguments

Online augmentation is enabled for experiments B and D. For experiments A and C, online augmentation keys are set to neutral values. `final_epochs` is used when available.


In [7]:
def online_aug_args_from_config(config: dict) -> dict:
    online = config.get("online_augmentation", {})
    return {
        "hsv_h": float(online.get("hsv_h", 0.0)),
        "hsv_s": float(online.get("hsv_s", 0.0)),
        "hsv_v": float(online.get("hsv_v", 0.0)),
        "degrees": float(online.get("degrees", 0.0)),
        "translate": float(online.get("translate", 0.0)),
        "scale": float(online.get("scale", 0.0)),
        "perspective": float(online.get("perspective", 0.0)),
        "fliplr": float(online.get("fliplr", 0.0)),
        "flipud": float(online.get("flipud", 0.0)),
        "mosaic": float(online.get("mosaic", 0.0)),
        "mixup": float(online.get("mixup", 0.0)),
        "close_mosaic": int(online.get("close_mosaic", 0)),
    }


def online_aug_off_args(keys: dict) -> dict:
    return {key: (0 if key == "close_mosaic" else 0.0) for key in keys}


FINAL_EPOCHS = int(training_config.get("final_epochs", training_config["epochs"]))
DRY_RUN_FINAL = bool(training_config.get("dry_run_final", False))
OVERWRITE_FINAL_RUN = bool(training_config.get("overwrite_final_run", False))

online_train_args = online_aug_args_from_config(augmentation_config)
uses_online_augmentation = selected_experiment in ONLINE_AUG_EXPERIMENTS

final_train_args = {
    "epochs": FINAL_EPOCHS,
    "imgsz": int(training_config["imgsz"]),
    "batch": int(training_config["batch"]),
    "device": training_config["device"],
    "workers": int(training_config["workers"]),
    "patience": int(training_config["patience"]),
    "seed": int(training_config["seed"]),
    "amp": bool(training_config.get("amp", False)),
    "dry_run": DRY_RUN_FINAL,
    "selected_experiment": selected_experiment,
}
final_train_args.update(
    online_train_args
    if uses_online_augmentation
    else online_aug_off_args(online_train_args)
)

run_name = f"final_{Path(selected_model).stem}_{selected_experiment}"

print(f"Online augmentation enabled: {uses_online_augmentation}")
display(pd.DataFrame([final_train_args]))

Online augmentation enabled: True


,epochs,imgsz,batch,device,workers,patience,seed,amp,dry_run,selected_experiment,...,hsv_v,degrees,translate,scale,perspective,fliplr,flipud,mosaic,mixup,close_mosaic
0,100,640,24,0,8,20,42,True,False,exp_D_full_pipeline,...,0.35,5.0,0.1,0.5,0.0003,0.5,0.0,1.0,0.05,10


## 7. Train Final Model

This trains one final model under `runs/final_model/`. If the run name already exists and overwrite is off, a safe suffix is used.


In [8]:
final_training_result = train_final_model(
    selected_model=selected_model,
    data_yaml=selected_data_yaml,
    output_dir=FINAL_RUNS_DIR,
    run_name=run_name,
    train_args=final_train_args,
    overwrite=OVERWRITE_FINAL_RUN,
)

display(pd.DataFrame([final_training_result]))

New https://pypi.org/project/ultralytics/8.4.58 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.20 torch-2.13.0.dev20260517+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\runs\final_model\_data_exp_D_full_pipeline_resolved.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.3, hsv_v=0.35, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, 

,selected_model,selected_experiment,data_yaml,resolved_data_yaml,run_name,run_dir,best_weights_path,last_weights_path,training_status,notes,error_message,precision,recall,map50,map50_95,fitness,training_time_seconds,model_size_mb
0,yolov8n.pt,exp_D_full_pipeline,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,final_yolov8n_exp_D_full_pipeline,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,trained,final training completed,,0.93992,0.81936,0.87769,0.60914,None,400.007,5.959452


## 8. Copy Final Weights

For convenience, copy `best.pt` and `last.pt` into `weights/final/` after successful training. These are generated artifacts and should remain ignored by Git.


In [9]:
weights_copy_rows = []
for key, filename in [
    ("best_weights_path", "best.pt"),
    ("last_weights_path", "last.pt"),
]:
    source = Path(str(final_training_result.get(key, "")))
    target = FINAL_WEIGHTS_DIR / filename
    row = {
        "weight_type": key,
        "source": str(source),
        "target": str(target),
        "status": "skipped",
        "notes": "",
    }
    if final_training_result.get("training_status") == "dry_run":
        row["notes"] = "dry run; no weights produced"
    elif source.exists():
        shutil.copy2(source, target)
        row["status"] = "copied"
    else:
        row["notes"] = "source weights not found"
    weights_copy_rows.append(row)

weights_copy_df = pd.DataFrame(weights_copy_rows)
display(weights_copy_df)

,weight_type,source,target,status,notes
0,best_weights_path,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
1,last_weights_path,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,


## 9. Evaluate Once On Untouched Test Set

This is the only final model-selection pipeline notebook that evaluates on the test split. Do not use the test metrics below to change architecture, augmentation, or training settings.


In [10]:
best_weights = Path(str(final_training_result.get("best_weights_path", "")))
if not best_weights.exists() and (FINAL_WEIGHTS_DIR / "best.pt").exists():
    best_weights = FINAL_WEIGHTS_DIR / "best.pt"

if final_training_result.get("training_status") == "dry_run":
    final_test_result = {
        "status": "skipped",
        "split": "test",
        "weights_path": str(best_weights),
        "data_yaml": str(selected_data_yaml),
        "precision": None,
        "recall": None,
        "map50": None,
        "map50_95": None,
        "fitness": None,
        "validation_save_dir": "",
        "class_metrics": [],
        "error_message": "dry run; final test evaluation skipped",
    }
else:
    final_test_result = evaluate_final_model(
        weights_path=best_weights,
        data_yaml=selected_data_yaml,
        imgsz=int(training_config["imgsz"]),
        device=training_config["device"],
        split="test",
        output_dir=FIGURES_DIR / "final_test_eval",
    )

display(
    pd.DataFrame([{k: v for k, v in final_test_result.items() if k != "class_metrics"}])
)

Ultralytics 8.4.51  Python-3.10.20 torch-2.13.0.dev20260517+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 903.855.1 MB/s, size: 1326.1 KB)
val: Scanning C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\experiments\exp_D_full_pipeline\test\labels... 50 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 252.2it/s 0.2s.4s
val: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\experiments\exp_D_full_pipeline\test\images\test_00045.png: 1 duplicate labels removed
val: New cache created: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\experiments\exp_D_full_pipeline\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.6it/s 2.5s0.4ss
                   all         50        684      0

,status,split,weights_path,data_yaml,precision,recall,map50,map50_95,fitness,validation_save_dir,error_message
0,evaluated,test,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.88273,0.734193,0.806942,0.478385,0.478385,C:\Github\smart-factory-safety-monitoring-syst...,


## 10. Save Optional Test Predictions

Prediction images are for qualitative reporting. They should not be used to tune thresholds or revise the final model after seeing test outcomes.


In [11]:
if final_training_result.get("training_status") == "dry_run":
    prediction_result = {
        "status": "skipped",
        "weights_path": str(best_weights),
        "source_dir": str(dataset_dirs["test_images"]),
        "output_dir": str(FIGURES_DIR / "final_test_predictions"),
        "num_source_images": 0,
        "error_message": "dry run; prediction export skipped",
    }
else:
    prediction_result = save_final_predictions(
        weights_path=best_weights,
        source_dir=dataset_dirs["test_images"],
        output_dir=FIGURES_DIR / "final_test_predictions",
        imgsz=int(training_config["imgsz"]),
        device=training_config["device"],
        conf=float(training_config.get("final_prediction_conf", 0.25)),
    )

display(pd.DataFrame([prediction_result]))

Results saved to C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\figures\final_test_predictions


,status,weights_path,source_dir,output_dir,num_source_images,error_message
0,saved,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,50,


## 11. Export Final Model

Export to ONNX by default. TensorRT or other formats can be added to `final_export_formats`, but export failures are logged instead of stopping the notebook.


In [12]:
export_formats = list(training_config.get("final_export_formats", ["onnx"]))
if final_training_result.get("training_status") == "dry_run":
    export_report = pd.DataFrame(
        [
            {
                "format": export_format,
                "status": "skipped",
                "source_weights": str(best_weights),
                "exported_path": "",
                "copied_path": "",
                "error_message": "dry run; export skipped",
            }
            for export_format in export_formats
        ]
    )
else:
    export_report = export_final_model(
        weights_path=best_weights,
        output_dir=FINAL_WEIGHTS_DIR,
        formats=export_formats,
        imgsz=int(training_config["imgsz"]),
        device=training_config["device"],
    )

display(export_report)

Ultralytics 8.4.51  Python-3.10.20 torch-2.13.0.dev20260517+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\runs\final_model\final_yolov8n_exp_D_full_pipeline\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.10.20 environment at: c:\Users\ADMIN\.conda\envs\smart-factory-safety
Resolved 15 packages in 544ms
 Downloaded onnx
 Downloaded onnxruntime-gpu
Prepared 8 packages in 12.69s
Installed 8 packages in 3.39s
 + coloredlogs==15.0.1
 + flatbuffers==25.12.19
 + humanfrie

,format,status,source_weights,exported_path,copied_path,error_message
0,onnx,exported,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,


## 12. Save Final Reports

These reports capture the locked choices, final training status, one-time test metrics, optional class-wise metrics, and export outcomes.


In [13]:
summary_row = {
    "selected_model": selected_model,
    "selected_model_source": selected_model_source,
    "ablation_top_experiment": ablation_top_experiment,
    "selected_experiment": selected_experiment,
    "selected_experiment_reason": "full augmentation pipeline preferred for harder real-world CCTV robustness",
    "data_yaml": str(selected_data_yaml.relative_to(V2_ROOT)),
    "uses_online_augmentation": uses_online_augmentation,
    "final_epochs": FINAL_EPOCHS,
    "batch": int(training_config["batch"]),
    "workers": int(training_config["workers"]),
    "amp": bool(training_config.get("amp", False)),
    "training_status": final_training_result.get("training_status"),
    "run_dir": final_training_result.get("run_dir"),
    "best_weights_path": final_training_result.get("best_weights_path"),
    "last_weights_path": final_training_result.get("last_weights_path"),
    "notes": selection_note,
}

final_training_summary = pd.DataFrame([summary_row])
final_test_results = pd.DataFrame(
    [{k: v for k, v in final_test_result.items() if k != "class_metrics"}]
)
final_class_metrics = pd.DataFrame(final_test_result.get("class_metrics", []))
if final_class_metrics.empty:
    final_class_metrics = pd.DataFrame(
        columns=["class_id", "class_name", "precision", "recall", "map50", "map50_95"]
    )

final_training_summary.to_csv(REPORTS_DIR / "final_training_summary.csv", index=False)
final_test_results.to_csv(REPORTS_DIR / "final_test_results.csv", index=False)
final_class_metrics.to_csv(REPORTS_DIR / "final_class_metrics.csv", index=False)
export_report.to_csv(REPORTS_DIR / "final_export_report.csv", index=False)

print("Saved final reports:")
for filename in [
    "final_training_summary.csv",
    "final_test_results.csv",
    "final_class_metrics.csv",
    "final_export_report.csv",
]:
    print(f"- {(REPORTS_DIR / filename).relative_to(V2_ROOT)}")

Saved final reports:
- reports\training\final_training_summary.csv
- reports\training\final_test_results.csv
- reports\training\final_class_metrics.csv
- reports\training\final_export_report.csv


## 13. Final Checklist

- Final architecture came from Notebook 06/candidate reporting or explicit config fallback.
- Final experiment uses `exp_D_full_pipeline` by preference for harder real-world deployment conditions.
- The untouched test split was evaluated only after final choices were locked.
- Test results are saved for reporting only, not for another selection loop.
- Final weights and exports are under `weights/final/`.
